<style>
  .nvidia-banner {background: linear-gradient(100deg,#0b0b0b,#292929); color:white;
                  border-left:10px solid #76b900; padding:18px 22px; margin:8px 0 18px;}
  .nvidia-banner h1 {margin:0 0 6px; font-size:30px;}
  .task {border-left:6px solid #76b900; background:#f5f8f1; padding:12px 16px; margin:12px 0;}
  .checkpoint {border:1px solid #b8b8b8; border-radius:6px; padding:10px 14px; background:#fafafa;}
  .warning {border-left:6px solid #f2a900; background:#fff8e6; padding:12px 16px;}
  code {font-size: 0.92em;}
</style>

<div class="nvidia-banner">
  <h1>Module 3 — Agent-Guided nvMolKit Panel Design</h1>
  <div>ACS Fall 2026 · Agentic scientific workflow · 45–60 minutes</div>
</div>

## Mission

A scientific agent receives a panel-design objective rather than a function specification:

> **From a bounded ReFRAME candidate pool, design a 96-compound panel that is structurally diverse, retains broad physicochemical coverage, and is reproducible and auditable.**

Nemotron compares two defensible strategies and recommends one. You approve the scientific plan; the local controller renders its tested nvMolKit implementation, runs it, and validates the artifacts.


## Setup and roles

- **You:** scientific sponsor and reviewer. Start the agent, compare its strategies, approve one, and inspect the receipts.
- **Embedded scientific agent:** Nemotron interprets the data profile, compares strategies, recommends a plan, and audits the result.
- **nvMolKit:** accelerated executor for batch fingerprints, similarity, and clustering.
- **Python checks:** deterministic gatekeeper for rendered source, output shape, uniqueness, provenance, and numerical sanity.

The experience follows the original interactive demo: click **Start Agent**, review two strategy cards, choose bounded controls, then click **Approve Plan & Run Agent**. The controller-rendered code, execution receipt, validation, and final scientific audit remain visible in the notebook. No separate coding-agent application is required.

The hosted calls require network access and an NVIDIA Developer API key (`nvapi-`). Keep `workshop_llm_agent.py`, `module3_interactive_workflow.py`, `workshop_common.py`, and `data/reframe_teaching_snapshot.csv` beside the workshop notebooks.


In [ ]:
from pathlib import Path
import importlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from rdkit import Chem, DataStructs, RDLogger, rdBase
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

from workshop_common import add_descriptors, load_reframe

# Confirm that the notebook, bounded agent, and interactive controller match.
import workshop_llm_agent as _workshop_llm_agent

EXPECTED_WORKSHOP_AGENT_VERSION = "2026.08.14.22"
_workshop_llm_agent = importlib.reload(_workshop_llm_agent)
loaded_agent_version = getattr(
    _workshop_llm_agent, "WORKSHOP_AGENT_VERSION", "pre-version"
)
if loaded_agent_version != EXPECTED_WORKSHOP_AGENT_VERSION:
    raise RuntimeError(
        "workshop_llm_agent.py is out of date: "
        f"expected {EXPECTED_WORKSHOP_AGENT_VERSION}, found {loaded_agent_version}. "
        "Replace the agent file with the copy distributed with this notebook, "
        "then restart the kernel and run this cell again."
    )

import module3_interactive_workflow as _module3_interactive_workflow

EXPECTED_MODULE3_WORKFLOW_VERSION = "2026.08.14.4"
_module3_interactive_workflow = importlib.reload(_module3_interactive_workflow)
loaded_workflow_version = getattr(
    _module3_interactive_workflow, "MODULE3_WORKFLOW_VERSION", "pre-version"
)
if loaded_workflow_version != EXPECTED_MODULE3_WORKFLOW_VERSION:
    raise RuntimeError(
        "module3_interactive_workflow.py is out of date: "
        f"expected {EXPECTED_MODULE3_WORKFLOW_VERSION}, found {loaded_workflow_version}. "
        "Replace the controller file, restart the kernel, and run this cell again."
    )

from workshop_llm_agent import PanelDesignAgent, get_workshop_api_key
from module3_interactive_workflow import launch_interactive_panel_design

print(
    "Workshop agent:", loaded_agent_version,
    "| interactive workflow:", loaded_workflow_version,
)

RDLogger.DisableLog("rdApp.error")
SEED = 2026
np.random.seed(SEED)

NVMOLKIT_READY = False
NVMOLKIT_IMPORT_ERROR = None
try:
    import torch
    import nvmolkit
    from nvmolkit.clustering import fused_butina
    from nvmolkit.fingerprints import MorganFingerprintGenerator
    from nvmolkit.similarity import crossTanimotoSimilarity

    NVMOLKIT_READY = bool(torch.cuda.is_available())
except Exception as exc:
    NVMOLKIT_IMPORT_ERROR = repr(exc)

print(f"RDKit {rdBase.rdkitVersion}")
if NVMOLKIT_READY:
    print(f"nvMolKit {nvmolkit.__version__} | CUDA devices: {torch.cuda.device_count()}")
else:
    print("CPU teaching fallback active: nvMolKit GPU calls will be shown but evaluated with RDKit.")
    print("Reason:", NVMOLKIT_IMPORT_ERROR or "torch.cuda.is_available() is False")


In [ ]:
def fingerprint_tensor(result):
    """Return the CUDA torch tensor wrapped by an nvMolKit fingerprint result."""
    return result if isinstance(result, torch.Tensor) else result.torch()


def gpu_to_numpy(result):
    """Synchronize an nvMolKit result or CUDA tensor and return a host array."""
    if isinstance(result, torch.Tensor):
        return result.detach().cpu().numpy()
    return result.numpy()


def make_fingerprints(molecules, radius=2, fp_bits=1024):
    """Return nvMolKit CUDA fingerprints, or RDKit fingerprints in fallback mode."""
    if NVMOLKIT_READY:
        generator = MorganFingerprintGenerator(radius=radius, fpSize=fp_bits)
        return fingerprint_tensor(generator.GetFingerprints(list(molecules), num_threads=0))
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_bits)
    return generator.GetFingerprints(list(molecules), numThreads=0)


def tanimoto_matrix(first, second=None):
    """Return a host similarity matrix from the active backend."""
    if NVMOLKIT_READY:
        return gpu_to_numpy(crossTanimotoSimilarity(first, second))
    second = first if second is None else second
    return np.asarray(
        [DataStructs.BulkTanimotoSimilarity(query, second) for query in first],
        dtype=float,
    )


def nvmolkit_butina_labels(fingerprints, distance_cutoff):
    """Convert nvMolKit 0.5.0 fused-Butina clusters to one label per molecule."""
    clusters, _, centroids = fused_butina(
        fingerprints, cutoff=distance_cutoff, return_centroids=True
    )
    labels = np.full(len(fingerprints), -1, dtype=int)
    for cluster_id, members in enumerate(clusters):
        labels[list(members)] = cluster_id
    centroids = np.asarray(centroids, dtype=int)
    assert (labels >= 0).all(), "Every molecule must receive a cluster label"
    assert len(centroids) == len(clusters), "Centroid and cluster counts must match"
    return labels, centroids


def rdkit_butina_labels(fingerprints, distance_cutoff):
    """Run the conventional RDKit distance-vector and Butina workflow."""
    n_items = len(fingerprints)
    if n_items == 1:
        return np.array([0]), np.array([0])
    distances = []
    for row in range(1, n_items):
        distances.extend(
            DataStructs.BulkTanimotoSimilarity(
                fingerprints[row], fingerprints[:row], returnDistance=True
            )
        )
    clusters = Butina.ClusterData(
        distances, n_items, distance_cutoff, isDistData=True, reordering=True
    )
    labels = np.full(n_items, -1, dtype=int)
    centroids = []
    for cluster_id, members in enumerate(clusters):
        labels[list(members)] = cluster_id
        centroids.append(members[0])
    return labels, np.asarray(centroids, dtype=int)


def butina_labels(fingerprints, distance_cutoff=0.55):
    if NVMOLKIT_READY:
        return nvmolkit_butina_labels(fingerprints, distance_cutoff)
    return rdkit_butina_labels(fingerprints, distance_cutoff)


print("Shared ReFRAME helpers and Module 3 chemistry helpers are ready.")

### What the notebook computes

- **Morgan fingerprints** encode circular atom environments as a fixed-length bit vector. Radius changes how far each local environment extends; fingerprint length changes the collision budget.
- **Tanimoto similarity** compares two binary fingerprints. It is a structural-neighborhood measure, not a measurement of target binding or biological activity.
- **Butina clustering** groups molecules using a distance cutoff. Here, `distance = 1 - Tanimoto similarity`, so a cutoff of `0.55` corresponds to a similarity threshold of `0.45`.
- **2D molecular depictions** let you inspect the actual chemotypes behind the panel-level statistics. Drawings support chemical interpretation, but do not establish activity.

The notebook uses nvMolKit when a compatible NVIDIA GPU is available. Its CPU fallback exists only so the teaching narrative and checks remain inspectable on a non-GPU laptop; the workshop's accelerated exercises should report `NVMOLKIT_READY == True`.


In [ ]:
REQUESTED_POOL_SIZE = 10000
PANEL_SIZE = 96

# Keep CPU-only review bounded. The live GPU workshop uses the requested pool.
effective_pool_size = REQUESTED_POOL_SIZE if NVMOLKIT_READY else min(512, REQUESTED_POOL_SIZE)
candidate_pool = add_descriptors(
    load_reframe(
        effective_pool_size,
        anchor_terms=("imatinib", "linezolid", "ritonavir", "metformin"),
    )
)
print("Source:", candidate_pool.attrs["source"])
print(f"Candidate pool: {len(candidate_pool):,} compounds")
print("Backend:", "nvMolKit GPU" if NVMOLKIT_READY else "RDKit CPU fallback")


## Step 1 — Materialize a bounded agent workspace

This cell writes only workshop inputs and a mission brief. It does not select compounds. The embedded scientific agent proposes the strategy; the local controller confines execution to the printed directory and leaves machine-readable receipts.


In [ ]:
AGENT_WORKDIR = Path.cwd() / "module3_agent_workspace"
AGENT_WORKDIR.mkdir(parents=True, exist_ok=True)

input_columns = [
    "smile", "canonical_ikey", "name", "source", "source_id", "status",
    "reframedb_url", "MolWt", "cLogP", "TPSA", "HBD", "HBA", "RotB"
]
input_path = AGENT_WORKDIR / "reframe_candidates.csv"
candidate_pool[input_columns].to_csv(input_path, index=False)

MISSION = f"""
# Mission: design an auditable ReFRAME diversity panel

You are the scientific planning agent for an ACS chemistry workshop.
Read `reframe_candidates.csv` and propose how to design a panel of exactly {min(PANEL_SIZE, len(candidate_pool))}
unique compounds. Use seed {SEED}. The input is a bounded teaching sample, not the whole
ReFRAME collection. The local controller will render and execute the approved strategy.

## Scientific objective
Maximize structural diversity while retaining broad coverage of molecular weight, cLogP,
and TPSA. Prefer compounds whose status says they are available for follow-up. Explain how
your method balances large chemotypes against rare chemistry. You choose and justify the
Morgan radius, fingerprint size, clustering or diversity-selection method, and thresholds.

## Required technology
Use installed nvMolKit for Morgan fingerprints and either batch Tanimoto similarity
or fused Butina clustering.
Before writing code, inspect the installed workshop sources:
- /nvmolkit-brev-notebook/skills/nvmolkit/SKILL.md
- /.venv/lib/python3.12/site-packages/nvmolkit
These installed sources are authoritative because this workshop uses a newer build than
the public repository. Additional references:
- https://github.com/NVIDIA-BioNeMo/nvMolKit
- https://nvidia-bionemo.github.io/nvMolKit/

Current entry points:
- MorganFingerprintGenerator(radius, fpSize).GetFingerprints(mols)
- crossTanimotoSimilarity(first, second=None).numpy()
- clusters, cumulative_cluster_sizes, centroids = fused_butina(
    fingerprint_tensor, cutoff=<distance>, return_centroids=True
  )
  `clusters` is a list of member-index tuples, not a label vector. Convert each
  cluster's member indices into one integer label per input molecule.

## Required artifacts
1. `panel.csv`: one row per selected compound; preserve source columns and add
   `selection_reason`, `method_cluster`, and `selection_order`.
2. `report.json`: keys `seed`, `backend`, `parameters`, `candidate_count`, `panel_count`,
   `unique_ikeys`, `descriptor_quantiles`, `pairwise_similarity`, `cluster_coverage`,
   `limitations`, and `files`.
3. `analysis.py`: readable, rerunnable implementation with assertions and no notebook state.
4. Do not create extra artifacts; Step 4 provides a bounded 2D visual inspection.

## Acceptance criteria
- exactly {min(PANEL_SIZE, len(candidate_pool))} unique canonical_ikey values;
- every selection occurs in the input and retains its ReFRAME profile URL;
- all numeric similarities are finite and in [0, 1];
- parameters and seed are recorded; output is deterministic on the same environment;
- compare candidate vs panel descriptor quantiles and report pairwise-similarity summaries;
- no biological-activity, efficacy, safety, or clinical claims.

## Working style
First inspect the data profile and compare the two candidate strategy families. Wait for
sponsor approval before the controller renders and runs the chosen implementation. After
validation, inspect any surprising result and summarize tradeoffs with the output files.
""".strip()

(AGENT_WORKDIR / "MISSION.md").write_text(MISSION + "\n", encoding="utf-8")
print("Agent workspace:", AGENT_WORKDIR)
print("Input:", input_path)
display(Markdown("```text\n" + MISSION + "\n```"))


## Step 2 — Launch the interactive scientific agent

Run the next cell, enter the hosted NVIDIA Developer API key when prompted, then use the controls that appear:

1. Click **Start Agent**. Nemotron receives `MISSION.md` plus a compact deterministic profile of the candidate CSV and proposes exactly two strategies. It does not generate analysis code yet.
2. Compare the strategies and the recommendation. Decide how descriptor coverage and common-versus-rare chemotypes should be balanced.
3. Select a strategy and execution limit, then click **Approve Plan & Run Agent**.
4. Watch the controller render the approved strategy as tested nvMolKit code, execute it, validate its artifacts, and ask Nemotron for a scientific audit. Expand **View rendered analysis_attempt_1_rendered.py** to inspect the executable receipt.

Button callback failures remain in the interactive display rather than marking this notebook cell as failed. Nemotron owns scientific planning and interpretation; the local controller owns executable source. It runs without the hosted API key and accepts output only after deterministic artifact checks.

<div class="warning"><b>Execution boundary:</b> Static checks reject disallowed imports, obvious unsafe operations, and literal paths outside the workspace. Execution also has a timeout and no hosted key. This is a bounded workshop controller, not an operating-system security sandbox, so the displayed source remains an important sponsor receipt.</div>


In [ ]:
agent_api_key = get_workshop_api_key()
panel_agent = PanelDesignAgent(
    workdir=AGENT_WORKDIR,
    mission=MISSION,
    api_key=agent_api_key,
)

module3_workflow = launch_interactive_panel_design(
    panel_agent,
    expected_panel_size=min(PANEL_SIZE, len(candidate_pool)),
    max_revisions=0,
    timeout_seconds=480,
)


## Step 3 — Audit the sponsor-approved panel

Continue only after the interface reports **Agent workflow complete** or explains that the controller-rendered run did not pass.

This step loads the current result, checks it independently, and compares structural diversity with physicochemical coverage. A failed run deliberately selects the labeled reference baseline; stale files are never treated as a current successful result.

A good panel should lower redundant pairwise similarity without collapsing the descriptor range. There is no universal optimum; the sponsor must inspect and defend the tradeoff.


In [ ]:
agent_run = module3_workflow.agent_run
if agent_run is None:
    raise RuntimeError(
        "Complete the interactive workflow above before continuing: click Start Agent, "
        "review a strategy, and click Approve Plan & Run Agent."
    )

attempt_table = pd.DataFrame([
    {
        "attempt": attempt.number,
        "source_file": attempt.source_file,
        "seconds": attempt.elapsed_seconds,
        "passed": attempt.passed,
        "message": attempt.message,
        "implementation_summary": attempt.implementation_summary,
        "expected_tradeoffs": "; ".join(attempt.expected_tradeoffs),
    }
    for attempt in agent_run.attempts
])
display(attempt_table)

USE_AGENT_OUTPUT = agent_run.success
if USE_AGENT_OUTPUT:
    print("✓ The interactive agent produced independently validated artifacts.")
    print("Analysis:", agent_run.analysis_path)
    print("Trace:", agent_run.trace_path)
else:
    print("The controller-rendered analysis did not pass; the reference baseline will be used.")
    print("Inspect:", agent_run.trace_path)


### Load the current result

When the interactive run succeeds, this section loads its `panel.csv` and `report.json`. Otherwise it deliberately runs the tagged reference baseline. This makes recovery explicit and prevents stale workspace files from being mistaken for a successful current run.


In [ ]:
def reference_panel(records, panel_size=96, radius=2, fp_bits=1024, distance_cutoff=0.55):
    '''Select cluster centroids while interleaving three molecular-weight bands.'''
    available = records[records["status"].str.contains("available", case=False, na=False)].copy()
    if len(available) < panel_size:
        available = records.copy()
    available = available.reset_index(drop=True)

    fingerprints = make_fingerprints(available["_mol"].tolist(), radius, fp_bits)
    labels, centroids = butina_labels(fingerprints, distance_cutoff)
    available["method_cluster"] = labels
    cluster_sizes = available.groupby("method_cluster").size()

    centroid_frame = available.iloc[centroids].copy()
    centroid_frame["cluster_size"] = centroid_frame["method_cluster"].map(cluster_sizes)
    centroid_frame["mw_band"] = pd.cut(
        centroid_frame["MolWt"], [-np.inf, 300, 500, np.inf],
        labels=["<300", "300–500", ">500"]
    )

    band_queues = {
        band: group.sort_values(["cluster_size", "canonical_ikey"], ascending=[False, True]).index.tolist()
        for band, group in centroid_frame.groupby("mw_band", observed=True)
    }
    selected_indices = []
    bands = ["<300", "300–500", ">500"]
    while len(selected_indices) < min(panel_size, len(centroid_frame)):
        moved = False
        for band in bands:
            queue = band_queues.get(band, [])
            if queue and len(selected_indices) < panel_size:
                selected_indices.append(queue.pop(0))
                moved = True
        if not moved:
            break

    # A very low cutoff can yield fewer centroids than requested. Fill deterministically.
    if len(selected_indices) < min(panel_size, len(available)):
        remaining = [idx for idx in available.index if idx not in selected_indices]
        selected_indices.extend(remaining[: panel_size - len(selected_indices)])

    panel = available.loc[selected_indices[:panel_size]].copy().reset_index(drop=True)
    panel["selection_order"] = np.arange(1, len(panel) + 1)
    panel["selection_reason"] = "Butina centroid interleaved across molecular-weight bands"
    parameters = {
        "seed": SEED, "radius": radius, "fp_bits": fp_bits,
        "distance_cutoff": distance_cutoff,
        "backend": "nvmolkit" if NVMOLKIT_READY else "rdkit-fallback",
    }
    return panel, parameters


USE_AGENT_OUTPUT = bool(globals().get("USE_AGENT_OUTPUT", False))
agent_panel_path = AGENT_WORKDIR / "panel.csv"
agent_report_path = AGENT_WORKDIR / "report.json"
used_agent_output = USE_AGENT_OUTPUT

if used_agent_output:
    missing_artifacts = [
        path.name for path in (agent_panel_path, agent_report_path) if not path.exists()
    ]
    if missing_artifacts:
        raise FileNotFoundError(f"Missing agent artifacts: {missing_artifacts}")
    panel = pd.read_csv(agent_panel_path)
    report = json.loads(agent_report_path.read_text(encoding="utf-8"))
    panel = panel.merge(
        candidate_pool[["canonical_ikey", "_mol"]],
        on="canonical_ikey",
        how="left",
        validate="one_to_one",
    )
    parameters = report.get("parameters", {})
    print("Loaded artifacts from the current successful embedded-agent run.")
else:
    panel, parameters = reference_panel(
        candidate_pool, panel_size=min(PANEL_SIZE, len(candidate_pool))
    )
    print("USE_AGENT_OUTPUT is False; loaded the tagged reference baseline.")

display(panel[["selection_order", "name", "MolWt", "cLogP", "TPSA", "method_cluster", "selection_reason"]].head(12).round(2))


### Check panel membership and provenance

These checks run outside the agent's implementation. Independent validation matters because an agent can write tests that merely confirm its own assumptions.


In [ ]:
expected_panel_size = min(PANEL_SIZE, len(candidate_pool))
required_panel_columns = {
    "smile", "canonical_ikey", "name", "reframedb_url",
    "MolWt", "cLogP", "TPSA",
    "selection_reason", "method_cluster", "selection_order"
}
assert required_panel_columns.issubset(panel.columns)
assert len(panel) == expected_panel_size
assert panel["canonical_ikey"].nunique() == expected_panel_size
assert panel["canonical_ikey"].isin(candidate_pool["canonical_ikey"]).all()
assert panel["reframedb_url"].notna().all()
assert panel["_mol"].notna().all()
assert sorted(panel["selection_order"].astype(int)) == list(range(1, expected_panel_size + 1))
print("✓ Panel membership, uniqueness, provenance, and selection order pass independent checks.")


### Compare diversity and property coverage

The similarity summary measures structural redundancy, while the descriptor distributions show which parts of property space were retained or lost.


In [ ]:
panel_fps = make_fingerprints(panel["_mol"].tolist(), radius=2, fp_bits=1024)
panel_similarity = tanimoto_matrix(panel_fps)
assert panel_similarity.shape == (len(panel), len(panel))
assert np.isfinite(panel_similarity).all()
assert ((panel_similarity >= 0) & (panel_similarity <= 1)).all()

upper = panel_similarity[np.triu_indices(len(panel), k=1)]
nearest_other = np.where(np.eye(len(panel), dtype=bool), -np.inf, panel_similarity).max(axis=1)
diversity_summary = pd.Series({
    "pairwise_median": np.median(upper),
    "pairwise_95th_percentile": np.quantile(upper, 0.95),
    "median_nearest_neighbor": np.median(nearest_other),
    "maximum_pairwise_similarity": upper.max(),
}).round(3)
display(diversity_summary.to_frame("value"))

descriptor_columns = ["MolWt", "cLogP", "TPSA"]
quantiles = [0.05, 0.25, 0.50, 0.75, 0.95]
coverage_rows = []
for column in descriptor_columns:
    for quantile in quantiles:
        coverage_rows.append({
            "descriptor": column,
            "quantile": quantile,
            "candidate_pool": candidate_pool[column].quantile(quantile),
            "panel": panel[column].quantile(quantile),
        })
coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot(index=["descriptor", "quantile"], columns=[], values=["candidate_pool", "panel"]).round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, column in zip(axes, descriptor_columns):
    axis.hist(candidate_pool[column], bins=28, density=True, alpha=0.35, label="Candidates", color="#333333")
    axis.hist(panel[column], bins=18, density=True, alpha=0.65, label="Panel", color="#76b900")
    axis.set(title=column, xlabel=column, ylabel="Density")
axes[0].legend()
fig.suptitle("Does the diverse panel retain property coverage?", y=1.03)
plt.tight_layout()
plt.show()

# Put descriptor tails on a comparable scale for the sponsor discussion.
tail_coverage = coverage[coverage["quantile"].isin([0.05, 0.95])].copy()
iqr_by_descriptor = {
    descriptor: candidate_pool[descriptor].quantile(0.75) - candidate_pool[descriptor].quantile(0.25)
    for descriptor in descriptor_columns
}
tail_coverage["candidate_iqr"] = tail_coverage["descriptor"].map(iqr_by_descriptor)
tail_coverage["absolute_shift"] = (
    tail_coverage["panel"] - tail_coverage["candidate_pool"]
).abs()
tail_coverage["shift_in_candidate_iqr"] = np.where(
    tail_coverage["candidate_iqr"] > 0,
    tail_coverage["absolute_shift"] / tail_coverage["candidate_iqr"],
    np.nan,
)
tail_coverage = tail_coverage.sort_values("shift_in_candidate_iqr", ascending=False)
print("Largest normalized descriptor-tail change appears first")
display(tail_coverage.round(3))

selected_cluster_counts = panel.groupby("method_cluster", dropna=False).size().sort_values(ascending=False)
print(
    f"Panel members: {len(panel)} | represented method clusters: {len(selected_cluster_counts)} | "
    f"largest within-panel cluster contribution: {selected_cluster_counts.iloc[0]}"
)
if used_agent_output:
    print("Agent artifacts were used: inspect report.json for candidate-level cluster-size evidence.")
else:
    print("Reference baseline used: it prioritizes centroids of larger clusters within each molecular-weight band.")


<div class="checkpoint">
<b>Sponsor review</b><br>
1. Does the selection overrepresent large clusters or rare singletons?<br>
2. Which descriptor tail changed most, and is that acceptable for the imagined screen?<br>
3. Would a different fingerprint radius change what “diverse” means?<br>
4. What experimental constraint—solubility, plate layout, stock availability, assay interference—should enter the next iteration?
</div>


## Step 4 — Visualize the selected chemistry

Panel statistics summarize diversity, but chemists should also inspect the structures behind those numbers. The next cell draws a bounded page of 2D molecular structures with their selection order and key descriptors.

Change `GALLERY_PAGE` to move through the panel, or set `SORT_BY` to `MolWt`, `cLogP`, or `TPSA` to inspect a property tail. Then rerun the cell.

<div class="checkpoint"><b>Visual inspection</b><br>Do you see repeated scaffolds, unusual ring systems, highly polar structures, or compounds that make the numerical diversity summary easier to understand? A 2D drawing can reveal chemotypes, but it still cannot establish conformation, binding, or biological activity.</div>

In [ ]:
from rdkit.Chem import Draw

MOLECULES_PER_PAGE = 12
GALLERY_PAGE = 1             # Try 2, 3, ...
SORT_BY = "selection_order"  # Or: MolWt, cLogP, TPSA

allowed_sort_columns = {"selection_order", "MolWt", "cLogP", "TPSA"}
if SORT_BY not in allowed_sort_columns:
    raise ValueError(f"SORT_BY must be one of {sorted(allowed_sort_columns)}")

gallery_panel = panel.sort_values([SORT_BY, "canonical_ikey"]).reset_index(drop=True)
page_count = max(1, (len(gallery_panel) + MOLECULES_PER_PAGE - 1) // MOLECULES_PER_PAGE)
if not 1 <= GALLERY_PAGE <= page_count:
    raise ValueError(f"GALLERY_PAGE must be between 1 and {page_count}")

start = (GALLERY_PAGE - 1) * MOLECULES_PER_PAGE
gallery_rows = gallery_panel.iloc[start : start + MOLECULES_PER_PAGE].copy()
gallery_molecules = []
gallery_legends = []
for _, row in gallery_rows.iterrows():
    molecule = Chem.Mol(row["_mol"]) if row["_mol"] is not None else Chem.MolFromSmiles(row["smile"])
    if molecule is None:
        raise ValueError(f"Could not draw {row['name']}")
    gallery_molecules.append(molecule)
    short_name = str(row["name"])[:28]
    gallery_legends.append(
        f"{int(row['selection_order'])}. {short_name}\n"
        f"MW {row['MolWt']:.0f} | cLogP {row['cLogP']:.1f} | TPSA {row['TPSA']:.0f}"
    )

gallery = Draw.MolsToGridImage(
    gallery_molecules,
    legends=gallery_legends,
    molsPerRow=4,
    subImgSize=(260, 220),
    useSVG=True,
)
print(f"Showing page {GALLERY_PAGE} of {page_count}, sorted by {SORT_BY}")
display(gallery)
display(
    gallery_rows[["selection_order", "name", "MolWt", "cLogP", "TPSA", "reframedb_url"]]
    .reset_index(drop=True)
    .round(2)
)


## Final synthesis — grade the agent

Score each dimension from 0–2. If the embedded agent ran, use `analysis.py`, `agent_trace.json`, `panel.csv`, and `report.json` as receipts rather than grading its narrative alone:

| Dimension | 0 | 1 | 2 |
|---|---|---|---|
| Reproducibility | missing seed/receipts | partial | rerunnable with complete parameters |
| nvMolKit use | absent/incorrect | used but poorly batched | correct batch-oriented workflow |
| Chemical reasoning | unsupported | plausible but thin | explicit tradeoffs and sensitivity checks |
| Validation | self-authored only | basic independent checks | independent invariants plus surprise inspection |
| Scientific boundaries | overclaims | caveats present | claims tightly matched to evidence |

**Exit ticket:** In two sentences, state the panel-design decision you would defend and the next experiment or computation required before making a repurposing claim.


## Sources and scientific boundary

- [nvMolKit repository](https://github.com/NVIDIA-BioNeMo/nvMolKit)
- [nvMolKit documentation](https://nvidia-bionemo.github.io/nvMolKit/)
- [Installed workshop nvMolKit skill](../skills/nvmolkit/SKILL.md) — authoritative for this environment
- [reframeDb](https://reframedb.org/) and its public `reframe_smiles_list.csv` export

The ReFRAME export is used for teaching and should be handled under the site's current terms. Refresh it before delivery and do not treat availability status as evidence of clinical suitability. Fingerprints, descriptors, clusters, and 2D molecular drawings do **not** establish binding, activity, ADMET, efficacy, safety, synthesizability, conformation, or experimental structure.


## Answer key — agent and sponsor checkpoints

<details>
<summary><b>Reveal instructor key</b></summary>

### Embedded-agent plan and approval checkpoint

- A valid run requests an agent plan before the controller renders code, records the sponsor's chosen strategy, and leaves `analysis_attempt_1_rendered.py` plus `agent_trace.json`. Nemotron owns the scientific comparison and audit; the controller owns the tested executable scaffold. A successful status is not sufficient by itself; the independent notebook assertions remain authoritative.

- Two defensible strategies are: **(a)** Butina centroid selection interleaved across molecular-weight bands, which represents common chemotypes while broadening size coverage; and **(b)** greedy max-min selection in fingerprint space with an availability preference, which emphasizes separation between selected compounds. Either can be correct if parameters, tradeoffs, and deterministic tests are documented.
- A Butina centroid is locally representative because it has many neighbors within the cutoff. It does **not** optimize potency, safety, novelty, or clinical value.
- Property coverage should be measured explicitly—for example by candidate-versus-panel quantiles, histogram/bin coverage, or another declared distributional metric. “The plot looks broad” is insufficient by itself.
- The agent avoids biological overclaiming by describing the output as a structurally diverse screening panel and by requiring experimental evidence before discussing repurposing efficacy.

### Sponsor review

1. **Large clusters versus rare singletons:** For the reference baseline, the correct answer is that large clusters are favored: centroid candidates are ordered by cluster size within each molecular-weight band. Interleaving the bands protects molecular-weight coverage, but does not guarantee representation of rare singleton chemistry. For an agent-produced panel, use `report.json` and candidate-level cluster sizes; within-panel counts alone cannot prove whether rare input clusters were covered.

2. **Descriptor tail:** The first row of `tail_coverage` is the answer for the current run. It compares the 5th and 95th percentile shifts after dividing by each candidate descriptor’s IQR, making molecular weight, cLogP, and TPSA comparable despite different units. Whether the shift is acceptable is assay-specific; full credit requires connecting it to a declared screen constraint rather than applying an invented universal threshold.

3. **Fingerprint radius:** Yes. Radius changes which atom environments define similarity, so cluster membership, centroid choice, and the meaning of “diverse” may change. A strong workflow repeats selection at another defensible radius and reports panel overlap or diversity-metric sensitivity.

4. **Next experimental constraint:** Any well-justified assay-specific constraint can earn credit. Strong first additions are stock availability and solubility at the intended screening concentration, followed by plate controls/layout and known assay-interference liabilities. These constraints affect whether the computationally diverse panel can actually produce interpretable experimental data.

### Visual inspection

- The gallery should be used as a qualitative check on the numerical audit. Repeated cores may reveal local redundancy, while distinct ring systems, heteroatom patterns, sizes, and shapes help explain why fingerprints separated other compounds.
- Sorting by molecular weight, cLogP, or TPSA should make the corresponding property tail chemically visible. The drawing is not a substitute for the measured descriptor distribution, and apparent 2D differences do not prove different activity.

### Example exit ticket

> I would defend the selected panel as a reproducible, structurally diverse subset that preserves the measured molecular-weight, cLogP, and TPSA range under the stated fingerprint and clustering assumptions. Before making a repurposing claim, I would require a disease-relevant primary assay with orthogonal confirmation, counterscreens for interference/cytotoxicity, and follow-up exposure and safety evidence.

This is an example, not required wording. The essential distinction is **panel-design evidence now** versus **biological validation next**.

</details>
